#Phase 1 – Collecte automatique des offres d'emploi

Ce notebook implémente la phase de collecte des données pour le projet **RecruitAI**.
L'objectif est de collecter automatiquement des offres d'emploi depuis des plateformes
publiques afin de constituer un corpus exploitable pour les étapes d'analyse, de matching
et de génération de documents.




In [21]:
import requests
import json
import time
import os
import pandas as pd


In [28]:
# Variables d'environnement requises :
# GOOGLE_API_KEY
# GOOGLE_CSE_ID
from google.colab import userdata
os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')

GOOGLE_CSE_ID = os.getenv("GOOGLE_CSE_ID")

assert GOOGLE_API_KEY is not None, "❌ GOOGLE_API_KEY manquante"
assert GOOGLE_CSE_ID is not None, "❌ GOOGLE_CSE_ID manquante"

print("✅ Clés GCP chargées")


✅ Clés GCP chargées


##Classe BaseCollector

In [29]:
from abc import ABC, abstractmethod

class BaseCollector(ABC):
    """
    Interface abstraite pour tout collecteur d'offres d'emploi
    """

    @abstractmethod
    def collect(self, query, location, limit=10):
        pass


##GoogleJobsCollector (Indeed via Google)

In [30]:
class GoogleJobsCollector(BaseCollector):

    def __init__(self, api_key, cse_id):
        self.api_key = api_key
        self.cse_id = cse_id
        self.base_url = "https://www.googleapis.com/customsearch/v1"

    def collect(self, query, location, limit=10):
        offers = []
        start = 1

        while len(offers) < limit:
            params = {
                "key": self.api_key,
                "cx": self.cse_id,
                "q": f"site:indeed.com {query} {location}",
                "num": 10,
                "start": start
            }

            response = requests.get(self.base_url, params=params)
            data = response.json()

            if "items" not in data:
                break

            for item in data["items"]:
                offers.append({
                    "source": "Indeed (Google Jobs)",
                    "title": item.get("title"),
                    "company": item.get("displayLink"),
                    "location": location,
                    "description": item.get("snippet"),
                    "url": item.get("link")
                })

            start += 10
            time.sleep(1)  # respect quota GCP

        return offers[:limit]


##CollectorAgent (orchestrateur)


In [31]:
class CollectorAgent:

    def __init__(self, collectors):
        self.collectors = collectors

    def run(self, query, location, limit=20):
        all_offers = []

        for collector in self.collectors:
            try:
                offers = collector.collect(query, location, limit)
                all_offers.extend(offers)
            except Exception as e:
                print(f"[CollectorAgent] Erreur : {e}")

        return all_offers



##Initialisation de l’agent

In [32]:
google_collector = GoogleJobsCollector(
    api_key=GOOGLE_API_KEY,
    cse_id=GOOGLE_CSE_ID
)

collector_agent = CollectorAgent(
    collectors=[google_collector]
)


##Lancer la collecte

In [33]:
QUERY = "Data Scientist"
LOCATION = "Paris"
LIMIT = 20

offers = collector_agent.run(
    query=QUERY,
    location=LOCATION,
    limit=LIMIT
)

print(f"✅ {len(offers)} offres collectées")
offers[:2]


✅ 20 offres collectées


[{'source': 'Indeed (Google Jobs)',
  'title': 'Emplois : Data Scientist Biology, Paris (75) - 13 décembre 2025 ...',
  'company': 'fr.indeed.com',
  'location': 'Paris',
  'description': "Vous êtes à la recherche d'un emploi : Data Scientist Biology ? Il y en a 22 disponibles pour Paris (75) sur Indeed.com, le plus grand site d'emploi\xa0...",
  'url': 'https://fr.indeed.com/q-data-scientist-biology-l-paris-(75)-emplois.html'},
 {'source': 'Indeed (Google Jobs)',
  'title': 'Health Data Scientist, Paris (75) : plus de 75 emplois (13 décembre ...',
  'company': 'fr.indeed.com',
  'location': 'Paris',
  'description': "Vous êtes à la recherche d'un emploi : Health Data Scientist ? Il y en a 84 disponibles pour Paris (75) sur Indeed.com, le plus grand site d'emploi mondial.",
  'url': 'https://fr.indeed.com/q-health-data-scientist-l-paris-(75)-emplois.html'}]

In [34]:
df = pd.DataFrame(offers)
df


,source,title,company,location,description,url
0,Indeed (Google Jobs),"Emplois : Data Scientist Biology, Paris (75) -...",fr.indeed.com,Paris,Vous êtes à la recherche d'un emploi : Data Sc...,https://fr.indeed.com/q-data-scientist-biology...
1,Indeed (Google Jobs),"Health Data Scientist, Paris (75) : plus de 75...",fr.indeed.com,Paris,Vous êtes à la recherche d'un emploi : Health ...,https://fr.indeed.com/q-health-data-scientist-...
2,Indeed (Google Jobs),"Staff Data Scientist, Paris (75) : plus de 25 ...",fr.indeed.com,Paris,Vous êtes à la recherche d'un emploi : Staff D...,https://fr.indeed.com/q-staff-data-scientist-l...
3,Indeed (Google Jobs),"Research Scientist, Paris (75) : plus de 100 e...",fr.indeed.com,Paris,"Data scientists play a key role at Kayrros, wo...",https://fr.indeed.com/q-research-scientist-l-p...
4,Indeed (Google Jobs),"Data Scientist, Paris (75) : plus de 500 emplo...",fr.indeed.com,Paris,Vous êtes à la recherche d'un emploi : Data Sc...,https://fr.indeed.com/q-data-scientist-l-paris...
5,Indeed (Google Jobs),"Ai Research Scientist, Paris (75) : plus de 75...",fr.indeed.com,Paris,Data Scientist Internship. Pigment. Paris (75)...,https://fr.indeed.com/q-ai-research-scientist-...
6,Indeed (Google Jobs),"Ai Research Scientist, Paris (75) : plus de 75...",fr.indeed.com,Paris,"Follow industrial and academic research, occas...",https://fr.indeed.com/q-ai-research-scientist-...
7,Indeed (Google Jobs),Actuarial Data Scientist - Paris (75) - Indeed...,fr.indeed.com,Paris,Becoming an Actuarial Data Scientist is the id...,https://fr.indeed.com/viewjob?jk=7c9a2b382eedd00f
8,Indeed (Google Jobs),Internship Data Scientist : plus de 50 emplois...,fr.indeed.com,Paris,... data science;. Data Scientist Intern - Dig...,https://fr.indeed.com/q-internship-data-scient...
9,Indeed (Google Jobs),"Ai Research Scientist, Paris (75) : plus de 50...",fr.indeed.com,Paris,Be a proactive member in the choice and applie...,https://fr.indeed.com/q-ai-research-scientist-...


##Sauvegarde des données

In [35]:
with open("job_offers_raw.json", "w", encoding="utf-8") as f:
    json.dump(offers, f, indent=2, ensure_ascii=False)

print("💾 Données sauvegardées dans job_offers_raw.json")


💾 Données sauvegardées dans job_offers_raw.json


## Conclusion

Cette phase a permis de collecter automatiquement des offres d'emploi réelles
publiées sur Indeed en exploitant l'API Google Custom Search (GCP).

La collecte est assurée par un agent dédié, garantissant la modularité
et la possibilité d'étendre le système à d'autres sources à l'avenir.

Les données obtenues serviront de base pour la phase suivante :
l'analyse sémantique des offres à l'aide d'un modèle de langage (Gemini).
